# Data Quality — Detecting Missing Values

Energy time series often have gaps caused by sensor outages or communication
failures. The `data_quality` module provides functions to **detect**, **fill**,
and **visualize** these gaps.

This example uses synthetic 1-minute temperature data with intentional gaps.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## Create synthetic data with gaps

We generate 24 hours of 1-minute temperature data and remove two sections
to simulate sensor outages.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
idx = pd.date_range("2024-06-01", periods=1440, freq="min")
temp = 20 + 5 * np.sin(np.linspace(0, 2 * np.pi, 1440)) + np.random.normal(0, 0.3, 1440)
df = pd.DataFrame({"temperature": temp}, index=idx)

# Remove rows 200–250 and 800–900 to create gaps
df_with_gaps = df.drop(df.index[200:250]).drop(df.index[800:900])
print(f"Original: {len(df)} rows, With gaps: {len(df_with_gaps)} rows")
df_with_gaps.head()

## Detect gap durations

`calc_gap_duration` computes the time difference between consecutive rows
and a rolling median for comparison.

In [ ]:
from pyedautils.data_quality import calc_gap_duration

gaps = calc_gap_duration(df_with_gaps)
gaps[gaps["gapDuration"] > 60].head()

## Fill gaps with NaN

`fill_missing_values_with_na` inserts NaN rows at the expected timestamps
where data is missing.

In [ ]:
from pyedautils.data_quality import fill_missing_values_with_na

df_filled = fill_missing_values_with_na(df_with_gaps)
print(f"Before fill: {len(df_with_gaps)} rows, After fill: {len(df_filled)} rows")
print(f"NaN rows added: {df_filled['temperature'].isna().sum()}")

## Check NaN percentage

`calc_isna_percentage` gives a quick summary of data completeness.

In [ ]:
from pyedautils.data_quality import calc_isna_percentage

pct = calc_isna_percentage(df_filled, column="temperature")
print(f"Missing: {pct}%")

## Visualize missing values

`plot_missing_values` creates an interactive Plotly chart with red bands
highlighting the NaN regions.

In [ ]:
from pyedautils.data_quality import plot_missing_values

fig = plot_missing_values(df_filled, column="temperature", ylab="Temperature [°C]")
fig.show()